# Cleaning Downloaded Data from avian-flu

Author: Alexander Maksiaev

Purpose: Clean downloaded data from avian-flu, rename sequences according to convention

In [1]:
# Housekeeping

import os
import pandas as pd
import numpy as np
import dateutil 
from datetime import datetime
from collections import defaultdict 
import importlib
import utils  
importlib.reload(utils)
from utils import * 


In [2]:
# Dates
start_date = "07-05-2025"
end_date = "07-18-2025"
date_range = start_date + "--" + end_date

# Make sure you have the correct paths

# home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu/"
# downloads = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu/"
downloads = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
references = home + "references/"
originals = downloads + "Andersen/"
saved = originals + "saved/"
temp_files = originals + "temp/"
complete_files = originals + "complete/"
metadata_folder = originals + "avian-influenza/metadata/"

## Read Metadata 

In [5]:
# Read metadata

# Get metadata from GitHub repo
os.chdir(metadata_folder)
metadata = pd.read_csv("SraRunTable_automated_normalized.tsv", delimiter="\t")
print(len(metadata)) 

# If metadata_normalized.tsv is updated, merge to get collection dates
# os.chdir(saved)
# metadata_normalized = pd.read_csv("metadata_normalized.tsv", delimiter="\t") # Collection dates
# metadata = metadata.merge(metadata_normalized, how="outer")
print(metadata.columns)

# Find the name of the state sample was collected in
metadata["name_state"] = metadata["geo_loc_name"].apply(lambda x: x.split("/")[1] if len(x.split("/")[1]) > 0 else x.split("/")[0])

# Convert the dates to date format so we can compare
metadata["ReleaseDate"] = metadata["ReleaseDate"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y-%m-%d"))
metadata = metadata[metadata["ReleaseDate"] >= dateutil.parser.parse(start_date).strftime("%Y-%m-%d")] # Find only >= last date using Release Date from metadata 
metadata = metadata[metadata["ReleaseDate"] <= dateutil.parser.parse(end_date).strftime("%Y-%m-%d")] # Find only <= update date using Release Date from metadata

print(len(metadata)) 
display(metadata)

# os.chdir(saved)
# metadata = pd.read_csv("metadata_genbank_11-01-2021--07-04-2025.csv")

# metadata["Collection_Date_Specific"] = metadata_genbank["Collection_Date_Specific"] 

10228
Index(['Run', 'Assay Type', 'AvgSpotLen', 'Bases', 'BioProject', 'BioSample',
       'BioSampleModel', 'Bytes', 'Center Name', 'Collection_Date', 'Consent',
       'DATASTORE filetype', 'DATASTORE provider', 'DATASTORE region',
       'Experiment', 'geo_loc_name_country', 'geo_loc_name_country_continent',
       'geo_loc_name', 'Host', 'Instrument', 'isolate', 'Library Name',
       'LibraryLayout', 'LibrarySelection', 'LibrarySource', 'Organism',
       'Platform', 'ReleaseDate', 'create_date', 'version', 'Sample Name',
       'SRA Study', 'serotype', 'isolation_source', 'BioSample Accession',
       'is_retracted', 'retraction_detection_date_utc'],
      dtype='object')
329


,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,create_date,version,Sample Name,SRA Study,serotype,isolation_source,BioSample Accession,is_retracted,retraction_detection_date_utc,name_state
9899,SRR34491312,WGS,144.80,84195904,PRJNA1207547,SAMN49913376,Viral,33056457,USDA-NVSL,2024,...,2025-07-11 09:48:32,1,24-034551-014,SRP557452,NaN,CLOACAL/TRACHEAL SWAB POOL,SRS25752300,False,NaN,United States
9900,SRR34491313,WGS,145.74,113962328,PRJNA1207547,SAMN49913375,Viral,44629704,USDA-NVSL,2024,...,2025-07-11 09:48:34,1,24-034551-013,SRP557452,NaN,CLOACAL/TRACHEAL SWAB POOL,SRS25752298,False,NaN,United States
9901,SRR34491314,WGS,145.23,159309775,PRJNA1207547,SAMN49913374,Viral,62525676,USDA-NVSL,2024,...,2025-07-11 09:48:38,1,24-034551-010,SRP557452,NaN,CLOACAL/OROPHARYNGEAL SWAB POOL,SRS25752297,False,NaN,United States
9902,SRR34491315,WGS,130.34,149431286,PRJNA1207547,SAMN49913373,Viral,54098697,USDA-NVSL,2024,...,2025-07-11 09:48:33,1,24-034159-001,SRP557452,NaN,CLOACAL/OROPHARYNGEAL SWAB POOL,SRS25752295,False,NaN,United States
9903,SRR34491316,WGS,145.50,104717182,PRJNA1207547,SAMN49913372,Viral,31805328,USDA-NVSL,2024,...,2025-07-11 09:48:37,1,24-036728-001,SRP557452,NaN,CLOACAL/OROPHARYNGEAL SWAB POOL,SRS25752296,False,NaN,United States
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10223,SRR34542495,WGS,145.74,109679874,PRJNA980729,SAMN49975075,Viral,37013777,USDA-NVSL,2025,...,2025-07-15 12:22:51,1,25-019408-007,SRP441379,NaN,CLOACAL/OROPHARYNGEAL SWAB POOL,SRS25797466,False,NaN,United States
10224,SRR34542496,WGS,145.01,92718819,PRJNA980729,SAMN49975074,Viral,31474859,USDA-NVSL,2025,...,2025-07-15 12:22:53,1,25-019408-006,SRP441379,NaN,CLOACAL/OROPHARYNGEAL SWAB POOL,SRS25797465,False,NaN,United States
10225,SRR34542497,WGS,147.70,92271701,PRJNA980729,SAMN49975073,Viral,31462934,USDA-NVSL,2025,...,2025-07-15 12:22:49,1,25-019408-003,SRP441379,NaN,CLOACAL/OROPHARYNGEAL SWAB POOL,SRS25797464,False,NaN,United States
10226,SRR34542498,WGS,146.93,125871082,PRJNA980729,SAMN49975072,Viral,43138274,USDA-NVSL,2025,...,2025-07-15 12:22:50,1,25-019408-002,SRP441379,NaN,CLOACAL/OROPHARYNGEAL SWAB POOL,SRS25797463,False,NaN,United States


In [6]:
# Get list of genotypes

# os.chdir(references)

# genotypes_df = pd.read_excel("genotype_key.xlsx")

# genotypes = list(genotypes_df["Genotype"])

# print(genotypes)

genotypes = ["B3.13", "D1.1", "D1.3"]

# genotypes = ["B3.2"] #, "B3.2", "B3.6", "B3.7", "B3.5", "A3"]

### Naming convention ###
>A/[host]/[geo_loc_name]/[isolate]/[year]|[serotype: H5N1]|[geo_location]|[collection_date]|[host_type]|[genotype]

In metadata, we have: host, isolate, year

We need: geo_loc_name, collection_date, host_type, genotype

host = Host

geo_loc_name = genbank_mapping.tsv > genbank_name

geo_location = geo_loc_name (abbreviated)-country (abbreviated) e.g. USA-MD

isolate = isolate

collection date (primary) = Collection_Date

collection date = https://www.ncbi.nlm.nih.gov/genbank/ > BioSample (input: BioSample) > Nucleotide > [first result] > collection_date

serotype = H5N1 (hard-coded)

host type = animals_ref.csv (local)

genotype = genoflu_results.tsv > genotype

## Get genotype

In [7]:
# Get genotype from genoflu_results.tsv

os.chdir(metadata_folder)

genoflu_results = pd.read_csv("genoflu_results.tsv", delimiter="\t")
genoflu_results = genoflu_results.rename(columns={"sample" : "Run"}) # Rename so we can merge

metadata = metadata.merge(genoflu_results, on="Run", how="inner") # Add genoflu results to dataframe, excluding runs without results
metadata = metadata[metadata["Genotype"].isin(genotypes)]

print(len(metadata)) 
# display(metadata)

318


## Get specific geolocation

In [8]:
# Get specific geolocation and name_state from genbank_mapping.tsv
os.chdir(metadata_folder)
genbank_mapping = pd.read_csv("genbank_mapping.tsv", delimiter="\t")
genbank_mapping = genbank_mapping.rename(columns={"sra_run": "Run"}) # Rename so we can merge
genbank_mapping = genbank_mapping.drop_duplicates(subset="Run", keep="first") # Drop duplicates -- there are ~8 copies of each run
genbank_mapping["name_state"] = genbank_mapping["genbank_name"].apply(lambda x: x.split("/")[2]) # Get the name of the state

# print(genbank_mapping)

# Merge with metadata so that we can have specific geolocation
metadata_genbank = pd.concat([metadata, genbank_mapping], join="inner") # Exclude runs without geolocation

# print(metadata_genbank)

# Get geolocation for second state attribute

os.chdir(home + "references/")
state_ref = pd.read_csv("states_ref.csv")

# forbidden_chars = [", ", ": "] # List of characters to replace
# Format: USA-[state abbreviation], e.g. USA-MD
metadata_genbank["Geo_Location"] = metadata_genbank["name_state"].apply( # lambda x: state_ref.loc[state_ref["Abbreviation"] == x.split("/")[2], 'Country'].iloc[0] + "-" + x.split("/")[2] if x.split("/")[2] in state_ref["Abbreviation"].values else state_ref.loc[state_ref["State"] == x.split("/")[2].replace("_", " "), 'Country'].iloc[0] + "-" + state_ref.loc[state_ref["State"] == x.split("/")[2].replace("_", " "), 'Abbreviation'].iloc[0] if x.split("/")[2].replace("_", " ") in state_ref["State"].values else x.split("/")[2].replace(": ", "-"))
    
                                                        lambda x: 
                                                        # If "x" has the state abbreviation (e.g. "MD")
                                                        state_ref.loc[state_ref["Abbreviation"].str.contains('|'.join(x.replace(": ", ",").replace(" ", "_").split(',')), regex=True), 'Country'].iloc[0] 
                                                        + "-" + 
                                                        x
                                                        if state_ref["Abbreviation"].str.contains("|".join((x.replace(": ", ",").replace(" ", "_").split(','))), regex=True).any()
                                                        # If "x" has the full state name (e.g. "Maryland")
                                                        else state_ref.loc[state_ref['State'].str.contains('|'.join(x.replace(": ", ",").replace(" ", "_").split(',')), regex=True), 'Country'].iloc[0]
                                                        + "-" + 
                                                        state_ref.loc[state_ref['State'].str.contains('|'.join(x.replace(": ", ",").replace(" ", "_").split(',')), regex=True), 'Abbreviation'].iloc[0] 
                                                        if state_ref["State"].str.contains("|".join((x.replace(": ", ",").replace(" ", "_").split(','))), regex=True).any() 
                                                        # If "x" has neither the state abbreviation nor the full state name
                                                        else 
                                                        "USA")

# Rename variable back to metadata as we merge metadata and metadata_genbank
metadata = metadata.merge(metadata_genbank, on="Run")

display(metadata)

display(len(set(list(metadata["BioSample"]))))

,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,date,File Name,Genotype,"Genotype List Used, >=98.0%",Genotype Sample Title List,Genotype Percent Match List,Genotype Mismatch List,Genotype Average Depth of Coverage List,name_state_y,Geo_Location
0,SRR34491312,WGS,144.80,84195904,PRJNA1207547,SAMN49913376,Viral,33056457,USDA-NVSL,2024,...,2025-07-17_06-31-37,SRR34491312.fa,D1.1,"PA:am4, NP:am13, NS:ea3, MP:ea3, PB1:ea3, PB2:...","am4:24-030039-001:PA, am13:24-030039-001:NP, e...","99.72%, 99.60%, 99.05%, 99.90%, 99.30%, 99.78%...","6, 6, 8, 1, 16, 5, 8, 5",Ran on FASTA - No Coverage Report,United States,USA
1,SRR34491313,WGS,145.74,113962328,PRJNA1207547,SAMN49913375,Viral,44629704,USDA-NVSL,2024,...,2025-07-17_06-31-31,SRR34491313.fa,D1.1,"PB1:ea3, NP:am13, MP:ea3, NA:am4N1, HA:ea3, PB...","ea3:22-013001-001:PB1, am13:24-030039-001:NP, ...","99.43%, 99.93%, 100.00%, 99.04%, 99.47%, 99.91...","13, 1, 0, 10, 9, 2, 6, 6",Ran on FASTA - No Coverage Report,United States,USA
2,SRR34491314,WGS,145.23,159309775,PRJNA1207547,SAMN49913374,Viral,62525676,USDA-NVSL,2024,...,2025-07-17_06-31-31,SRR34491314.fa,D1.1,"PA:am4, HA:ea3, NS:ea3, PB1:ea3, PB2:am24, NP:...","am4:24-030039-001:PA, ea3:22-013001-001:HA, ea...","99.60%, 99.47%, 99.28%, 99.43%, 99.87%, 99.93%...","7, 9, 6, 13, 3, 1, 0, 8",Ran on FASTA - No Coverage Report,United States,USA
3,SRR34491315,WGS,130.34,149431286,PRJNA1207547,SAMN49913373,Viral,54098697,USDA-NVSL,2024,...,2025-07-17_06-31-31,SRR34491315.fa,D1.1,"PB1:ea3, NA:am4N1, NP:am13, NS:ea3, PB2:am24, ...","ea3:22-013001-001:PB1, am4N1:24-030039-001:NA,...","99.43%, 99.23%, 99.93%, 99.28%, 99.91%, 100.00...","13, 8, 1, 6, 2, 0, 5, 9",Ran on FASTA - No Coverage Report,United States,USA
4,SRR34491316,WGS,145.50,104717182,PRJNA1207547,SAMN49913372,Viral,31805328,USDA-NVSL,2024,...,2025-07-17_06-31-31,SRR34491316.fa,D1.1,"MP:ea3, HA:ea3, NA:am4N1, NS:ea3, PA:am4, NP:a...","ea3:22-013001-001:MP, ea3:22-013001-001:HA, am...","100.00%, 99.47%, 99.20%, 99.05%, 99.58%, 99.93...","0, 9, 9, 8, 9, 1, 20, 3",Ran on FASTA - No Coverage Report,United States,USA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
313,SRR34542495,WGS,145.74,109679874,PRJNA980729,SAMN49975075,Viral,37013777,USDA-NVSL,2025,...,2025-07-19_06-19-09,SRR34542495.fa,D1.1,"HA:ea3, NA:am4N1, NS:ea3, PB1:ea3, NP:am13, PA...","ea3:22-013001-001:HA, am4N1:24-030039-001:NA, ...","99.30%, 99.04%, 99.17%, 99.33%, 99.67%, 99.49%...","12, 11, 7, 12, 5, 9, 5, 0",Ran on FASTA - No Coverage Report,United States,USA
314,SRR34542496,WGS,145.01,92718819,PRJNA980729,SAMN49975074,Viral,31474859,USDA-NVSL,2025,...,2025-07-19_06-19-09,SRR34542496.fa,D1.1,"MP:ea3, NS:ea3, NA:am4N1, PB1:ea3, HA:ea3, PA:...","ea3:22-013001-001:MP, ea3:22-013001-001:NS, am...","100.00%, 99.05%, 99.23%, 99.06%, 99.24%, 99.49...","0, 8, 8, 17, 13, 9, 4, 5",Ran on FASTA - No Coverage Report,United States,USA
315,SRR34542497,WGS,147.70,92271701,PRJNA980729,SAMN49975073,Viral,31462934,USDA-NVSL,2025,...,2025-07-19_06-19-09,SRR34542497.fa,D1.1,"NP:am13, HA:ea3, MP:ea3, NS:ea3, PA:am4, PB1:e...","am13:24-030039-001:NP, ea3:22-013001-001:HA, e...","99.53%, 99.24%, 100.00%, 99.05%, 99.49%, 99.34...","7, 13, 0, 8, 9, 12, 4, 14",Ran on FASTA - No Coverage Report,United States,USA
316,SRR34542498,WGS,146.93,125871082,PRJNA980729,SAMN49975072,Viral,43138274,USDA-NVSL,2025,...,2025-07-19_06-19-09,SRR34542498.fa,D1.1,"PB2:am24, PA:am4, MP:ea3, NP:am13, PB1:ea3, NA...","am24:24-030039-001:PB2, am4:24-030039-001:PA, ...","99.74%, 99.49%, 99.90%, 99.53%, 99.39%, 99.13%...","6, 9, 1, 7, 11, 10, 8, 12",Ran on FASTA - No Coverage Report,United States,USA


318

## Collection Dates

If date is N/A, try finding it first. If a csv file of saved dates (NOT metadata_normalized.tsv) are available, do NOT run the next cell. Comment it out and run the cell after. 

In [ ]:

# # Get all dates
# metadata["Collection_Date_Specific"] = metadata["BioSample"].apply(lambda x: search_collection_date(x, metadata) if "-" not in x else x) # Real dates have dashes
# # Convert dates to date format
# try:
#     metadata["Collection_Date_Specific"] = metadata["Collection_Date_Specific"].apply(lambda x: dateutil.parser.parse(x, default=datetime(1, 1, 2000), fuzzy=True))
# except:
#     print("Unable to parse collection date.")

If saved dates are available, un-comment and run the next cell

In [8]:
# # Upload saved data -- if doing this, make sure the above cell is commented out
# os.chdir(temp_files)
# metadata_genbank = pd.read_csv("metadata_genbank_11-01-2021--06-13-2025.csv")

# # Get only updated dates

# def find_unknown_dates(x, df):
#     try:
#         date = metadata_genbank[metadata_genbank["BioSample"] == x]["Collection_Date"].values[0]
#         date = dateutil.parser.parse(date, default=datetime(2000, 1, 1), fuzzy=True) # Default is January 1st, 2000
#         if date.day == dateutil.parser.parse("1/1/2000").day and date.month == dateutil.parser.parse("1/1/2000").month: # If the date autocompleted to default 1/1
#             print("year only")
#             date = search_collection_date(x, df)
#         print("Success", date)
#     except:
#         date = search_collection_date(x, df) # If it's not parseable as a date

#     # Ensure that the date is converted to date format
#     try:
#         # date = dateutil.parser.parse(date, default=datetime(2000, 1, 1), fuzzy=True)
#         # print("yay")
#         if date.day == dateutil.parser.parse("1/1/2000").day and date.month == dateutil.parser.parse("1/1/2000").month: # If the date autocompleted to default 1/1
#             date = date.year.strftime("%Y") # Preserve only year
#         else: # If actual date
#             date = date.strftime("%Y-%m-%d")
#         print("Successfully parsed", date)
#     except:
#         print("Unable to parse date.")

#     return date # "If" statement in lambda function will search for the "just year" values

# updated_dates = metadata["BioSample"].apply(lambda x: find_unknown_dates(x, metadata)) # Update unknown dates, if possible
# metadata["Collection_Date_Specific"] = updated_dates

In [ ]:
# # If all dates are in the spreadsheet, replace all dates in dataframe

# os.chdir(saved)
# metadata_genbank = pd.read_csv("metadata_genbank_11-01-2021--07-04-2025.csv")

# metadata["Collection_Date_Specific"] = metadata_genbank["Collection_Date_Specific"] 

In [10]:
# # Save the above so we don't have to do it again
# os.chdir(temp_files)
# metadata.to_csv("metadata_genbank_" + date_range + ".csv")

# Get years from collection dates
metadata["years"] = metadata["Collection_Date"].apply(lambda x: dateutil.parser.parse(x, default=datetime(2000, 1, 1), fuzzy=True).year if x == x else str(x)) # Get year only from collection date

print(metadata["Collection_Date"])

0      2024
1      2024
2      2024
3      2024
4      2024
       ... 
313    2025
314    2025
315    2025
316    2025
317    2025
Name: Collection_Date, Length: 318, dtype: object


## Get host type

In [12]:
# create a mask, where is True if the host does not exist
mask = metadata["Host"].isna()

# choose between the original value and split isolate using the mask
metadata["Host"] = np.where(mask, 
                            metadata["isolate"].apply(lambda x: 
                                                      x if x != x # If NaN
                                                      or "/" not in x # If split isolate doesn't exist 
                                                      or len(x.split("/")) < 2 # If split isolate is too short
                                                      else x.split("/")[1]), metadata["Host"]) # Provided that we have a long enough isolate with "/" in them, get the host

metadata["Host"] = metadata["Host"].apply(lambda x: x.lower() if x == x else x) # make sure all characters are lowercase

# Create animals ref if needed
unique_animals_all = sort_animals_andersen(metadata)

# Flatten unique_animals_all
every_unique_animal = []
for animal in unique_animals_all:
    every_unique_animal.append(animal)

# print(every_unique_animal)

unique_animals_set = list(set(every_unique_animal)) # Get rid of duplicates

os.chdir(home + "references/")
animals_ref = pd.read_csv("animals_ref.csv") # Upload animals ref

# If animal not in ref1, put in ref2
common_animals = []
# Check if animals in unique_animals_set are in ref1
for animal in unique_animals_set:
    for col in animals_ref.columns:
        if animal in animals_ref[col].values and type(animal) == str: # If animal exists in dataframe and isn't NaN
            common_animals.append(animal)

# If not in ref1, make a list of the new animals
different_animals = []
for animal in unique_animals_set:
    if animal not in common_animals:
        different_animals.append(animal)

print(different_animals)

# Add to dataframe
animals_df = animals_ref
# Make different_animals same length as dataframe, if shorter
if len(different_animals) < len(animals_df):
    number_of_times_to_add_nan = len(animals_df) - len(different_animals)
    for i in range(number_of_times_to_add_nan):
        different_animals.append(float('nan'))
# Unlikely for different_animals to be longer than the dataframe, but just in case
else:
    number_of_times_to_add_nan = len(different_animals) - len(animals_df)
    for i in range(number_of_times_to_add_nan):
        empty_rows = pd.DataFrame(np.nan, index=range(number_of_times_to_add_nan), columns=animals_df.columns)
        animals_df = pd.concat([animals_df, empty_rows], ignore_index=True)

animals_df["new"] = (different_animals)

print(animals_df)

animals_df.to_csv("animals_ref_to_sort.csv") # Make sure name is different to avoid overwriting the first reference 

print(metadata["Host"])
# print(metadata["isolate"])

[]
            wild_avian domestic_avian               cattle        feline  \
0     great_horned_owl       flamingo            dairy_cow           cat   
1         common_raven       pheasant               cattle  domestic_cat   
2        cooper's_hawk         turkey  cattle milk product     feral_cat   
3         coopers_hawk        chicken          bovine_milk        feline   
4              peafowl          goose              bovine   domestic-cat   
..                 ...            ...                  ...           ...   
788  lesser snow goose            NaN                  NaN           NaN   
789   ring-necked duck            NaN                  NaN           NaN   
790      laughing gull            NaN                  NaN           NaN   
791       caspian tern            NaN                  NaN           NaN   
792        eared grebe            NaN                  NaN           NaN   

      other_mammal       human         other  new  
0       deer mouse  washington  

In [13]:
# Get animals from animal reference
os.chdir(references)
animals_ref = pd.read_csv("animals_ref.csv")
fix_animals_andersen(metadata, animals_ref) # Get host type

## Make names using all the attributes we collected

In [15]:
# Make names

metadata = metadata.fillna("") # Make sure the entire name does not become "NaN"

names = ">" + metadata["Run"] + "|" + "A/" + metadata["Host"].apply(lambda x: x.replace(" ", "_")) + "/" + metadata["name_state_y"] + "/" + metadata["Sample Name"] + "/" + metadata["years"].apply(lambda x: str(x)) + "|H5N1|" + metadata["Geo_Location"] + "|" + metadata["Collection_Date"].apply(lambda x: x if "-" not in x else str(dateutil.parser.parse(x, default=datetime(2000, 1, 1)).strftime("%Y")) if dateutil.parser.parse(x, default=datetime(2000, 1, 1)).month == datetime(2000, 1, 1).month and dateutil.parser.parse(x, default=datetime(2000, 1, 1)).day == datetime(2000, 1, 1).day else dateutil.parser.parse(x, default=datetime(2000, 1, 1)).strftime("%Y-%m-%d")) + "|" + metadata["Host_Type"] + "|" + metadata["Genotype"]

metadata["Name"] = names

display(metadata["Name"])

0      >SRR34491312|A/blue-winged_teal/United States/...
1      >SRR34491313|A/blue-winged_teal/United States/...
2      >SRR34491314|A/blue-winged_teal/United States/...
3      >SRR34491315|A/northern_shoveler/United States...
4      >SRR34491316|A/gull/United States/24-036728-00...
                             ...                        
313    >SRR34542495|A/mallard_duck/United States/25-0...
314    >SRR34542496|A/mallard_duck/United States/25-0...
315    >SRR34542497|A/chukar/United States/25-019408-...
316    >SRR34542498|A/chukar/United States/25-019408-...
317    >SRR34542499|A/chukar/United States/25-019408-...
Name: Name, Length: 318, dtype: object

In [16]:
# Drop duplicate runs 
metadata = metadata.drop_duplicates(subset="Run", keep="first")

In [17]:
print(metadata)
# metadata.to_csv("metadata_test.csv")

             Run Assay Type  AvgSpotLen      Bases    BioProject  \
0    SRR34491312        WGS      144.80   84195904  PRJNA1207547   
1    SRR34491313        WGS      145.74  113962328  PRJNA1207547   
2    SRR34491314        WGS      145.23  159309775  PRJNA1207547   
3    SRR34491315        WGS      130.34  149431286  PRJNA1207547   
4    SRR34491316        WGS      145.50  104717182  PRJNA1207547   
..           ...        ...         ...        ...           ...   
313  SRR34542495        WGS      145.74  109679874   PRJNA980729   
314  SRR34542496        WGS      145.01   92718819   PRJNA980729   
315  SRR34542497        WGS      147.70   92271701   PRJNA980729   
316  SRR34542498        WGS      146.93  125871082   PRJNA980729   
317  SRR34542499        WGS      148.06   96369330   PRJNA980729   

        BioSample BioSampleModel     Bytes Center Name Collection_Date  ...  \
0    SAMN49913376          Viral  33056457   USDA-NVSL            2024  ...   
1    SAMN49913375        

## Make FASTA files

In [18]:
# Get information to create the fasta files

fasta_folder = originals + "avian-influenza/fasta/"

os.chdir(fasta_folder)

segments = ["PB2", "PB1", "PA", "NS", "NP", "NA", "MP", "HA"]
pairs = []
fasta_files = {}

# Create pairs of genotypes and segments, e.g. B3.13_HA
for genotype in genotypes: # ["B3.13", "D1.1"]:
    for segment in segments:
        pair = genotype + "_" + segment
        pairs.append(pair)

for pair in pairs:
    fasta_files[pair] = [] # List to hold fasta files

for run in metadata["Run"].values: # For each run 
    for dirpath, dirs, files in os.walk(fasta_folder): # Find the fasta file
        for file in files:
            file_name = os.path.join(dirpath, file) # Get file name
            # print(file_name)
            if run in file_name: # Note that there will be ~8 files total with that run name
                # Make a fasta file and put it in the list
                with open(file_name) as f:
                    lines = f.readlines()
                    sequence = lines[1] 
                    # Each run/segment pair has one sequence -- it's placed into a file with other run/segment pairs with the same segment and genotype
                    header = metadata[metadata["Run"] == run].loc[:, "Name"].values[0]
                    genotype = metadata[metadata["Run"] == run].loc[:, "Genotype"].values[0]
                    # print(header)
                    # print(genotype)
                    # break 
                    segment = file_name.split("_")[-2]
                    # Find the pair that corresponds to 
                    pair_name = genotype + "_" + segment
                    this_specific_fasta = []
                    for pair in pairs:
                        # print(pair)
                        # print(pair_name)
                        if pair_name == pair:
                            this_specific_fasta.append(header)
                            this_specific_fasta.append(sequence)
                            fasta_files[pair].append(this_specific_fasta)
                f.close()
        break 

In [19]:
# Create fasta files 

os.chdir(originals + "complete/")
names = []
for pair in fasta_files.keys():
    if len(fasta_files[pair]) > 0: # If this isn't empty
        output_path = originals + "complete/" + pair + "_" + date_range + "_andersen.fasta"

        output_file = open(output_path, "w")
        for item in fasta_files[pair]:
            # for item in item:
            # item = fasta_files[pair]
            try:
                name = str(item[0].values[0]) # See if this is one we didn't have a collection date for
            except:
                name = str(item[0])
            print(name)
            names.append(name)
            # First is header, second is sequence
            # print(value)
            output_file.write(name + "\n")
            output_file.write(item[1])
        output_file.close()

print(len(names)/8)

>SRR34425685|A/cattle/United States/25-018598-003/2025|H5N1|USA|2025|cattle|B3.13
>SRR34425686|A/cattle/United States/25-018149-003/2025|H5N1|USA|2025|cattle|B3.13
>SRR34425687|A/cattle/United States/25-018149-002/2025|H5N1|USA|2025|cattle|B3.13
>SRR34425688|A/cattle/United States/25-018149-001/2025|H5N1|USA|2025|cattle|B3.13
>SRR34425689|A/cattle/United States/25-018144-003/2025|H5N1|USA|2025|cattle|B3.13
>SRR34425690|A/cattle/United States/25-018143-007/2025|H5N1|USA|2025|cattle|B3.13
>SRR34425691|A/cattle/United States/25-015735-001/2025|H5N1|USA|2025|cattle|B3.13
>SRR34425692|A/cattle/United States/25-015290-001/2025|H5N1|USA|2025|cattle|B3.13
>SRR34542513|A/cattle/United States/25-019211-001/2025|H5N1|USA|2025|cattle|B3.13
>SRR34542514|A/cattle/United States/25-019210-002/2025|H5N1|USA|2025|cattle|B3.13
>SRR34542515|A/cattle/United States/25-019210-001/2025|H5N1|USA|2025|cattle|B3.13
>SRR34542516|A/cattle/United States/25-019018-006/2025|H5N1|USA|2025|cattle|B3.13
>SRR34542517|A/c

In [ ]:
# # Fix dates

# os.chdir(home + "references/")
# state_ref = pd.read_csv("states_ref.csv")

# os.chdir(complete_files + "test/")

# # Function to prepare dataframes
# def fasta_df_og(file_name, states_ref):

#     fasta = pd.DataFrame()
#     headers = []
#     isolate_ids = []
#     isolate_names = []
#     subtypes = []
#     # segments = []
#     collection_dates = []
#     sequences = []
#     host_types = []
#     species = []
#     identifiers = []
#     genotypes = []
#     name_states = []
#     with open(file_name) as f:
#         lines = f.readlines()
#         for num, line in enumerate(lines):
#             # print(line)
#             if line[0] == ">": # If it's a header
#                 if line[1:].strip() not in headers: # And the previous line is not a header we've seen before
#                     header = line[1:].strip() # Remove the ">"
#                     # print(header)
#                     split_header = header.split("|")
#                     if len(header.split("|")) > 6:
#                             identifier = header.split("|")[0]
#                             identifiers.append(identifier)
#                             split_first_header = split_header[1].split("/")
#                     else:
#                         identifiers.append("unknown")
#                         split_first_header = split_header[0].split("/")
#                     # print(split_first_header)
#                     # print(split_header)
#                     headers.append(header) 
#                     name_states.append(split_first_header[2].replace("_", " "))
#                     isolate_ids.append(split_first_header[3])
#                     isolate_names.append(split_header[-6]) # We'll need to extract data from this too
#                     # print(split_header[2].split("_")[-1])
#                     subtypes.append(split_header[-5])  # Get only H5N1
#                     genotypes.append(split_header[-1])
#                     # segments.append(split_header[].split("_")[-1])
#                     host_types.append(split_header[-2])
#                     species.append(split_first_header[1])
#                     # if split_header[4] == "2024-01-01":
#                     #     collection_dates.append("2024") # No samples were collected 1/1/2024, these are all unknown 
#                     # elif split_header[4] == "2025-01-01":
#                     #     collection_dates.append("2025")
#                     # else: 
#                     collection_dates.append(split_header[-3].split("_")[-1])
#                     if num < len(lines): # If we're not at the last line
#                         # for i, l in enumerate(lines[num + 1:]):
#                         i = num
#                         sequence = ""
#                         # print(lines[i])
#                         # print(lines[i + 1])
#                         while i < len(lines) - 1 and lines[i + 1][0] != ">": # While the next line is part of a sequence
#                             sequence = sequence + lines[i + 1].strip()
#                             i += 1
#                         sequences.append(sequence) # Add next line to sequences
#         f.close()

#     # Create columns for data frame 
#     fasta["Header"] = headers
#     fasta["Isolate_Id"] = isolate_ids
#     fasta["Isolate_Name"] = isolate_names
#     fasta["Subtype"] = subtypes
#     fasta["name_state"] = name_states
#     # fasta["Segment"] = segments
#     # Geo_Location is more complicated
#     fasta["Geo_Location"] = fasta["name_state"].apply(lambda x: 
                                                      
#                                                         # If "x" has the state abbreviation (e.g. "MD")
#                                                         states_ref.loc[states_ref["Abbreviation"].str.contains('|'.join(x.replace(": ", ",").replace(" ", "_").split(',')), regex=True), 'Country'].iloc[0] 
#                                                         + "-" + 
#                                                         x
#                                                         if states_ref["Abbreviation"].str.contains("|".join((x.replace(": ", ",").replace(" ", "_").split(','))), regex=True).any()
#                                                         # If "x" has the full state name (e.g. "Maryland")
#                                                         else states_ref.loc[states_ref['State'].str.contains('|'.join(x.replace(": ", ",").replace(" ", "_").split(',')), regex=True), 'Country'].iloc[0]
#                                                         + "-" + 
#                                                         states_ref.loc[states_ref['State'].str.contains('|'.join(x.replace(": ", ",").replace(" ", "_").split(',')), regex=True), 'Abbreviation'].iloc[0] 
#                                                         if states_ref["State"].str.contains("|".join((x.replace(": ", ",").replace(" ", "_").split(','))), regex=True).any() 
#                                                         # If "x" has neither the state abbreviation nor the full state name
                                                        
#                                                         else "USA")
#     fasta["Date Collected"] = collection_dates
#     fasta["Species"] = species
#     fasta["Host_Type"] = host_types
#     fasta["Genotype"] = genotypes
#     fasta["Sequence"] = sequences
#     if len(identifiers) == len(fasta):
#         fasta["Identifier"] = identifiers
    
#     return fasta

# original_fasta_dfs = {}

# for dirpath, dirs, files in os.walk(complete_files):
#     for file in files:
#         file_name = os.path.join(dirpath, file)
#         if ".fasta" in file_name: 
#             new_df = fasta_df_og(file_name, state_ref)
#             # original_fasta_dfs[file_name] = fasta_file
#             new_df["Header"] = ">" + new_df["Identifier"] + "|" + new_df["Isolate_Name"] + "|" + "H5N1" + "|" + new_df["Geo_Location"] + "|" + new_df["Date Collected"].apply(lambda x: str(dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).year) if dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).month == dateutil.parser.parse("2000-01-01").month and dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).day == dateutil.parser.parse("2000-01-01").day else dateutil.parser.parse(str(x)).strftime("%Y-%m-%d")) + "|" + new_df["Host_Type"] + "|" + new_df["Genotype"]
#             output_path = file_name

#             output_file = open(output_path, "w")
#             for index, row in new_df.iterrows():
#                 name = row["Header"]
#                 item = row["Sequence"]
#                 # for item in item:
#                 # item = fasta_files[pair]
#                 # try:
#                 #     name = str(item[0].values[0]) # See if this is one we didn't have a collection date for
#                 # except:
#                 #     name = str(item[0])
#                 # print(name)
#                 # names.append(name)
#                 # First is header, second is sequence
#                 # print(value)
#                 output_file.write(name + "\n")
#                 output_file.write(item + "\n")
#             output_file.close()
#             # print(fasta_file)
#             # print(new_df)
            
#     break 



                                                 Header  \
0     >SRR28752446|A/blackbird/USA/24-008354-001-ori...   
1     >SRR28752447|A/cattle/USA/24-009108-005-origin...   
2     >SRR28752448|A/cattle/USA/24-009108-004-origin...   
3     >SRR28752449|A/cattle/USA/24-009108-003-origin...   
4     >SRR28752450|A/cattle/USA/24-009108-002-origin...   
...                                                 ...   
4798  >SRR34270201|A/cattle/USA/25-015005-001/2025|H...   
4799  >SRR34270202|A/cattle/USA/25-015002-001/2025|H...   
4800  >SRR34270203|A/cattle/USA/25-014358-003/2025|H...   
4801  >SRR34270204|A/cattle/USA/24-033523-004/2024|H...   
4802  >SRR34270205|A/cattle/USA/24-030030-004/2024|H...   

                  Isolate_Id                                 Isolate_Name  \
0     24-008354-001-original  A/blackbird/USA/24-008354-001-original/2024   
1     24-009108-005-original     A/cattle/USA/24-009108-005-original/2024   
2     24-009108-004-original     A/cattle/USA/24-009108-004-